# Example 2 — Full Pipeline with Google Earth Engine

This notebook fetches NDVI directly from Sentinel-2 and Landsat using GEE,
then estimates the crop stage.

### Prerequisites

1. A GEE account — sign up at https://earthengine.google.com
2. `earthengine-api` installed: `pip install earthengine-api`
3. Authenticate once: `earthengine authenticate` in your terminal

Then call `ee.Initialize()` at the top of your script **before** importing `gee_fetch`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import ee
ee.Initialize()   # <-- must come before gee_fetch import

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from gee_fetch import fetch_ndvi, run_crop_stage_from_gee
from crop_stage import smooth_daily_interpolate_ndvi, estimate_stage_adaptive

## 1. Define a polygon

`fetch_ndvi` accepts any of: file path, GeoDataFrame, shapely Geometry,
GeoJSON dict, or a list of `[lon, lat]` coordinate pairs.

In [ ]:
# Example: a small field polygon as a list of [lon, lat] pairs (WGS84)
polygon = [
    [-77.85, 35.62],
    [-77.84, 35.62],
    [-77.84, 35.61],
    [-77.85, 35.61],
    [-77.85, 35.62],   # close the ring
]

START_DATE = "2023-03-01"
END_DATE   = "2023-11-30"

## 2. Fetch NDVI from GEE

In [ ]:
ndvi_df = fetch_ndvi(
    polygon,
    start_date=START_DATE,
    end_date=END_DATE,
    poly_name="example_field",
    buffer_m=-10,     # inset 10 m to avoid boundary pixels
)

print(f"Retrieved {len(ndvi_df)} observations")
ndvi_df.head()

## 3. Smooth and estimate stage

In [ ]:
df_smooth = smooth_daily_interpolate_ndvi(ndvi_df)
result = estimate_stage_adaptive(
    df_smooth["NDVI_smooth"].to_numpy(),
    dates=df_smooth["date"],
)

for k, v in result.items():
    print(f"  {k:20s}: {v}")

## 4. Visualise

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for sensor, grp in ndvi_df.groupby("sensor"):
    ax.scatter(grp["date"], grp["NDVI"], alpha=0.5, s=35, label=sensor)

ax.plot(df_smooth["date"], df_smooth["NDVI_smooth"], color="steelblue", lw=2, label="Smoothed")
ax.axhline(result["Lower_threshold"], color="orange", ls="--",
           label=f"Lower ({result['Lower_threshold']:.2f})")
ax.axhline(result["Upper_threshold"], color="green",  ls="--",
           label=f"Upper ({result['Upper_threshold']:.2f})")

if result.get("Peak_date"):
    ax.axvline(result["Peak_date"], color="purple", ls=":",
               label=f"Peak ({result['Peak_date'].strftime('%d-%b-%Y')})")

ax.set_ylim(0, 1)
ax.set_ylabel("NDVI")
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"Stage: {result['Stage']} — {result['Stage_description']}")
plt.tight_layout()
plt.show()

## 5. Batch: estimate stage for many fields

`run_crop_stage_from_gee` fetches and estimates all polygons in a GeoDataFrame
in parallel using a thread pool.

In [ ]:
import geopandas as gpd

# Load your field boundaries (replace with your own file)
# gdf = gpd.read_file("my_fields.geojson")

# results = run_crop_stage_from_gee(
#     gdf,
#     id_col="field_id",       # column with unique field identifiers
#     lookback_days=180,       # days of NDVI history to fetch
#     max_workers=8,
# )
# results[["field_id", "crop_stage", "stage_description", "peak_date", "days_since_peak"]]